In [24]:
import os
HOME = "/Users/chandler/Downloads/CVCP"  # Set your project directory path
import ultralytics
from ultralytics import YOLO # Import YOLO class. This class is used to create a YOLOv8 model
from IPython.display import display, Image
from roboflow import Roboflow
import torch
from tqdm import tqdm
from ultralytics.nn.tasks import DetectionModel
import torch.serialization
from torch.nn.modules.container import Sequential

print(f"Project directory: {HOME}")
HOME

Project directory: /Users/chandler/Downloads/CVCP


'/Users/chandler/Downloads/CVCP'

_______________________________________________________________________________________________

_______________________________________________________________________________________________

# Training the model
modify the /data_path/ yourselve

modify data.yaml file as well

In [ ]:
%cd {HOME}
HOME
data_path= "/Users/chandler/Downloads/CVCP/datasets_yolocp/weed-crop-aerial-2/data.yaml"
model = YOLO("yolov8n.yaml")
results = model.train(data= data_path, epochs=50, imgsz=640, plots=True)

#/Users/chandler/Downloads/CVCP/datasets_yolocp/weed-crop-aerial-2/data.yaml

#Model Fine-Tuning


In [ ]:
# Fine-tune YOLOv8n on the weed/crop dataset
%cd {HOME}


model = YOLO(f'{HOME}/(example)runs/detect/train/weights/best.pt')  # Replace with your model path

# Fine-tune the model with your validation set
# Note: Typically you'd use a separate training set, but if you want to fine-tune with validation data:
results = model.train(
    data=f'{HOME}/datasets_yolocp/weed-crop-aerial-2/data.yaml',  # Your dataset configuration file
    epochs=20,                       # Fewer epochs for fine-tuning
    imgsz=640,                       # Image size
    batch=16,                        # Batch size (adjust based on GPU memory)
    lr0=0.0001,                      # Lower initial learning rate for fine-tuning
    lrf=0.01,                        # Final learning rate factor
    warmup_epochs=3,                 # Warmup epochs
    patience=10,                     # Early stopping patience
    save=True,                       # Save checkpoints
    device=0,                        # GPU device (use 'cpu' for CPU)
    workers=8,                       # Number of worker threads
    project='fine_tuning',           # Project name
    name='yolov8n_finetuned',        # Run name
    exist_ok=True,                   # Overwrite existing project
    pretrained=True,                 # Use pretrained weights (already loaded)
    optimizer='AdamW',               # Optimizer choice
    verbose=True,                    # Verbose output
    seed=42,                         # Random seed for reproducibility
    deterministic=True,              # Deterministic mode
    close_mosaic=10,                 # Disable mosaic augmentation in last N epochs
    amp=True,                        # Automatic Mixed Precision
    fraction=1.0,                    # Use full dataset (reduce if needed)
)

# Validate the fine-tuned model
validation_results = model.val()

# Print results
print("\n=== Fine-tuning Results ===")
print(f"mAP50: {validation_results.box.map50:.4f}")
print(f"mAP50-95: {validation_results.box.map:.4f}")
print(f"Precision: {validation_results.box.mp:.4f}")
print(f"Recall: {validation_results.box.mr:.4f}")

# Save the final model
model.save('fine_tuned_yolov8n.pt')
print("\nFine-tuned model saved as 'fine_tuned_yolov8n.pt'") # disabled to avoid augmentation-related errors



This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.


/Users/chandler/Downloads/CVCP


FileNotFoundError: [Errno 2] No such file or directory: '/Users/chandler/Downloads/CVCP/(example)runs/detect/weights/best.pt'

_______________________________________________________________________________________________

# Model Evaluation
When we are analysing how well YOLO is at predicting the contents of an image, there are several metrics we can use.
The most important ones are the **training loss** and the **validation loss**. The lower these values are, the better your algorithm is at predicting data. 

In [ ]:
%cd {HOME}
Image(filename=f'{HOME}/runs/detect/train/results.png', width=600)

# Furthermore, here is the F-1 Curve
The F-1 curve tells us the overall performance of our model. It is particularly insightful because it **accounts for underrepresented classes**.
Imagine you have a thousand pictures of dogs and five of cats. You might have high accuracy if you always output dogs, but your F1 score will reflect this issue. 

In [ ]:
Image(filename=f'{HOME}/runs/detect/train/F1_curve.png', width=600)

_______________________________________________________________________________________________

## Testing the model
Previously, the model only saw pictures in the **train** folder. Now, we will show it the pictures in the **test** folder, pictures the model has never seen before. Based on how good the model's performance is with the test images, we can have an idea of what the model's performance with data in the real world will be.

## Test our model

In [ ]:
# Load a model
%cd {HOME}
model_path=f"{HOME}/runs/detect/train/weights/best.pt"
model_2 = YOLO(model_path)  # our trained YOLOv8n model

# Run batched inference on a list of images
results_2 = model_2(test1) 

# Process results list
for result in results_2:
    result.show()  # display to screen